### Parâmetros necessários
- Comprimento de Bjerrum:	$l_B$	m
- Comprimento de Debye:	$\kappa^{-1}$	m
- Volume excluído não eletrostático:	$\omega$ Adimensional
- Comprimento do segmento estatístico:	$b$	m
- Número de segmentos	$N$	Adimensional
- Fração de cargas positivas:	$f$	Adimensional
- Fração de cargas negativas:	$g$	Adimensional

**Parâmetro adimensional para o volume físico**
$$
\omega^* =
\omega+
\frac{4\pi l_B(f-g)^2}{b^3\kappa^2}
-
\frac{\pi l_B^2(f+g)^2}{b^3\kappa}
$$

**Comprimento de Bjerrum**
$$
l_B=\frac{e^2}{4\pi\varepsilon_0\varepsilon_r k_BT}
$$

**Comprimento de Debye**

Para uma solução com força iônica $I$, expressa em mol/L:
$$
\kappa^2 =
\frac{2e^2N_A(1000)I}
{\varepsilon_0\varepsilon_r k_BT}
$$

**Fator de expansão** <br>
Utilizado para calcular como a cadeia se expande ou se compacta em relação à sua conformação ideal.
$$
\alpha^5-\alpha^3 =
\frac{4}{3}
\left(\frac{3}{2\pi}\right)^{3/2}
N^{1/2}\omega^*
$$

### Implementação 

In [1]:

import numpy as np

from scipy.constants import (
    elementary_charge as e,
    epsilon_0,
    Boltzmann as k_B,
    Avogadro as N_A
)


# Funções auxiliares

def bjerrum_length(T, epsilon_r):
    return e**2 / (
        4 * np.pi * epsilon_0 * epsilon_r * k_B * T
    )


def debye_length(T, epsilon_r, ionic_strength_M):
    kappa_sq = (
        2 * e**2 * N_A * 1000 * ionic_strength_M
        / (epsilon_0 * epsilon_r * k_B * T)
    )
    kappa = np.sqrt(kappa_sq)
    return 1 / kappa, kappa


# MODELO DE JOANNY (1999): ADSORÇÃO E INVERSÃO DE CARGA
def joanny_adsorption_layer(
    T=298.15,
    epsilon_r=78.5,
    ionic_strength_M=0.01,
    sigma_surface_nm2=-0.1,
    b_nm=0.7,
    N=100,
    f=0.3,
    chains_adsorbed_nm2=0.01,
    polymer_charge_sign=+1
):
    """
    Estimativa de adsorção de PPy sobre uma superfície carregada.

    Densidades de carga: cargas/nm².
    b: nm.
    N: número de monômeros.
    f: fração de monômeros carregados.
    """

    # Validação
    if T <= 0 or epsilon_r <= 0:
        raise ValueError("T e epsilon_r devem ser positivos.")
    if ionic_strength_M <= 0:
        raise ValueError("A força iônica deve ser positiva.")
    if b_nm <= 0 or N <= 0:
        raise ValueError("b e N devem ser positivos.")
    if not 0 < f <= 1:
        raise ValueError("f deve estar entre 0 e 1.")
    if chains_adsorbed_nm2 < 0:
        raise ValueError("A densidade de cadeias deve ser não negativa.")
    if polymer_charge_sign not in (-1, +1):
        raise ValueError("O sinal do polímero deve ser +1 ou -1.")
    if sigma_surface_nm2 == 0:
        raise ValueError("A superfície precisa ter carga não nula.")

    # Comprimentos eletrostáticos
    l_B = bjerrum_length(T, epsilon_r)

    lambda_D, kappa = debye_length(
        T, epsilon_r, ionic_strength_M
    )

    l_B_nm = l_B * 1e9
    lambda_D_nm = lambda_D * 1e9

    # Verificação da atração eletrostática
    sinal_superficie = np.sign(sigma_surface_nm2)

    if polymer_charge_sign == sinal_superficie:

        return {
            "Adsorção": False,
            "Regime": "Sem atração eletrostática",
            "Carga Superfície (cargas/nm²)": sigma_surface_nm2,
            "Carga do PPy (cargas/nm²)": 0.0,
            "Carga Resultante (cargas/nm²)": sigma_surface_nm2,
            "Carga Excedente (cargas/nm²)": 0.0,
            "Inversão de Carga": False,
            "Espessura da Camada (nm)": 0.0,
            "Comprimento de Debye (nm)": lambda_D_nm,
            "Bjerrum (nm)": l_B_nm,
            "kappa * delta": np.nan
        }

    # Espessura estimada da camada adsorvida
    sigma_abs = abs(sigma_surface_nm2)

    delta_nm = (
        b_nm**2 / (f * sigma_abs * l_B_nm)
    )**(1/3)

    # Critério de screening
    kappa_delta = kappa * delta_nm * 1e-9

    if kappa_delta < 1:
        regime = "Baixo screening"
    else:
        regime = "Screening significativo"

    # Carga fornecida pelo PPy
    sigma_polymer_nm2 = (
        polymer_charge_sign
        * chains_adsorbed_nm2
        * N
        * f
    )

    # Balanço de cargas
    sigma_resultante = (
        sigma_surface_nm2 + sigma_polymer_nm2
    )

    # Identificação da inversão
    inversao = (
        sigma_resultante != 0
        and np.sign(sigma_resultante) != sinal_superficie
    )

    carga_excedente = max(
        0,
        abs(sigma_polymer_nm2) - sigma_abs
    )

    return {
        "Adsorção": True,
        "Regime": regime,
        "Carga Superfície (cargas/nm²)": sigma_surface_nm2,
        "Carga do PPy (cargas/nm²)": sigma_polymer_nm2,
        "Carga Resultante (cargas/nm²)": sigma_resultante,
        "Carga Excedente (cargas/nm²)": carga_excedente,
        "Inversão de Carga": inversao,
        "Espessura da Camada (nm)": delta_nm,
        "Comprimento de Debye (nm)": lambda_D_nm,
        "Bjerrum (nm)": l_B_nm,
        "kappa * delta": kappa_delta
    }


# Modelagem do LbL (PPy / MoS2)
def simular_multicamadas_lbl(
    n_camadas,
    T,
    epsilon_r,
    ionic_strength_M,
    sigma_inicial_nm2,
    b_nm_ppy,
    N_ppy,
    f_ppy,
    chains_adsorbed_nm2,
    sigma_mos2_layer_nm2,
    espessura_mos2_nm
):
    """
    Simulação simplificada da deposição alternada de PPy e MoS2.
    A carga superficial é assinada.
    PPy:
        Carga positiva.
    MoS2:
        Carga negativa.
    A deposição de PPy utiliza a estimativa de adsorção
    de polieletrólito.
    """

    resultados = []

    sigma_atual = sigma_inicial_nm2

    for i in range(n_camadas):
        # Determinação do material
        if i % 2 == 0:
            material = "PPy"
            sinal_material = +1
        else:
            material = "MoS2"
            sinal_material = -1

        sigma_antes = sigma_atual

        # ADSORÇÃO DO PPy

        if material == "PPy":

            res = joanny_adsorption_layer(
                T=T,
                epsilon_r=epsilon_r,
                ionic_strength_M=ionic_strength_M,
                sigma_surface_nm2=sigma_atual,
                b_nm=b_nm_ppy,
                N=N_ppy,
                f=f_ppy,
                chains_adsorbed_nm2=chains_adsorbed_nm2,
                polymer_charge_sign=sinal_material
            )

            sigma_nova = res["Carga Resultante (cargas/nm²)"]
            espessura = res["Espessura da Camada (nm)"]
            regime = res["Regime"]
            adsorcao = res["Adsorção"]
            sigma_depositada = res["Carga do PPy (cargas/nm²)"]

        # ADSORÇÃO DE MoS2

        else:
            # Verifica se existe atração eletrostática
            if np.sign(sigma_atual) != sinal_material:

                # Contribuição efetiva de carga do MoS2
                sigma_depositada = (
                    sinal_material * sigma_mos2_layer_nm2
                )

                sigma_nova = sigma_atual + sigma_depositada
                espessura = espessura_mos2_nm
                regime = "Adsorção de nanosheet (simplificada)"
                adsorcao = True
            else:

                sigma_depositada = 0.0
                sigma_nova = sigma_atual
                espessura = 0.0
                regime = "Sem atração eletrostática"
                adsorcao = False

        # Armazenamento dos resultados

        resultados.append({

            "Camada": i + 1,
            "Material": material,
            "Adsorção": adsorcao,
            "Carga Antes (cargas/nm²)": sigma_antes,
            "Carga Depositada (cargas/nm²)": sigma_depositada,
            "Carga Após (cargas/nm²)": sigma_nova,
            "Espessura (nm)": espessura,
            "Regime": regime
        })

        # atualiza a superfície
        sigma_atual = sigma_nova

    return resultados


# Execução
if __name__ == "__main__":

    # PARÂMETROS DO SISTEMA
    params = {
        "T": 298.15,
        "epsilon_r": 78.5,
        "ionic_strength_M": 0.01,
        "sigma_inicial_nm2": -0.1,
        "b_nm_ppy": 0.7,
        "N_ppy": 100,
        "f_ppy": 0.3,
        "chains_adsorbed_nm2": 0.01,
        "sigma_mos2_layer_nm2": 0.3,
        "espessura_mos2_nm": 1.0
    }

    # CAMADA ÚNICA: PPy SOBRE MoS2
    resultado_unico = joanny_adsorption_layer(
        T=params["T"],
        epsilon_r=params["epsilon_r"],
        ionic_strength_M=params["ionic_strength_M"],
        sigma_surface_nm2=params["sigma_inicial_nm2"],
        b_nm=params["b_nm_ppy"],
        N=params["N_ppy"],
        f=params["f_ppy"],
        chains_adsorbed_nm2=params["chains_adsorbed_nm2"],
        polymer_charge_sign=+1
    )

    camadas = simular_multicamadas_lbl(
        n_camadas=5,
        **params
    )

    # ESPESSURA TOTAL DO FILME
    espessura_total = sum(
        c["Espessura (nm)"] for c in camadas
    )

import pandas as pd

df_resultados = pd.DataFrame(camadas)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

display(df_resultados)

with open("resultados_simulacao.txt", "w", encoding="utf-8") as arquivo:
    arquivo.write(df_resultados.to_string(index=False))

,Camada,Material,Adsorção,Carga Antes (cargas/nm²),Carga Depositada (cargas/nm²),Carga Após (cargas/nm²),Espessura (nm),Regime
0,1,PPy,True,-0.1,0.3,0.2,2.838791,Baixo screening
1,2,MoS2,True,0.2,-0.3,-0.1,1.000000,Adsorção de nanosheet (simplificada)
2,3,PPy,True,-0.1,0.3,0.2,2.838791,Baixo screening
3,4,MoS2,True,0.2,-0.3,-0.1,1.000000,Adsorção de nanosheet (simplificada)
4,5,PPy,True,-0.1,0.3,0.2,2.838791,Baixo screening
